# Spatial MS-AA soft-membership extension — revised

This notebook revises the first soft-membership extension after the quick-run
results.

The quick analysis showed three useful things:

1. **Group-level node-composition recovery was strong.**
2. MS-AA tracked the ordering of selective → mixed node organization, but
   tended to return a **more concentrated representation** than the generating
   memberships when overlap was strong.
3. The original single setting for participant heterogeneity did not establish
   reliable recovery of individual deviations.

This version therefore separates two questions.

## Part A — node mixing

Holding participant heterogeneity at the original moderate setting, vary the
true degree of node mixing:

- selective
- canonical overlap
- strongly mixed

The goal is to test recovery of the **organization** of overlapping node
participation and to characterize any systematic bias in the recovered degree
of mixing.

## Part B — participant heterogeneity

Hold node mixing at the canonical setting and vary **true between-participant
heterogeneity**:

- none
- moderate
- strong

This distinguishes three possibilities:

- MS-AA invents subject differences when none exist;
- modest true subject differences are below the recoverable scale;
- stronger differences become identifiable.

The participant-specific analysis focuses on **deviations from the group
profile** and **pairwise participant geometry**, rather than raw profile
correlations that can be inflated by the shared group structure.

## Empirical anchor

If the fitted empirical K=46 spatial-MS-AA solution is available, the notebook
also measures normalized node-membership entropy in the real data. This is used
only as a **descriptive geometric anchor** for the simulated overlap range; the
simulation is not tuned to reproduce an MS-AA performance result.

In [ ]:
from pathlib import Path
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import pdist

# ============================================================
# CONFIGURATION
# ============================================================

RUN_MODE = "paper"   # use "quick" first; switch to "paper" after inspection
RNG_SEED = 4242

HELPERS_DIR = "helpers"

OUTPUT_DIR = Path("simulation_soft_membership_revised_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIX_CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints_mixing"
MIX_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

HET_CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints_heterogeneity"
HET_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_DIR = OUTPUT_DIR / "details"
DETAIL_DIR.mkdir(parents=True, exist_ok=True)

MIX_RESULTS_CSV = OUTPUT_DIR / f"mixing_results_{RUN_MODE}.csv"
HET_RESULTS_CSV = OUTPUT_DIR / f"heterogeneity_results_{RUN_MODE}.csv"

# Same architecture as the canonical revision simulation.
N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
K_TRUE = 6

# Metastable temporal dynamics.
MEAN_DWELL = 12
TRANSITION_WIDTH = 4
SECONDARY_WEIGHT = 0.25
AR1_RHO = 0.85

# Deliberately off-model variation.
GLOBAL_GAIN_SD = 0.15
SIGNED_DEVIATION_SD = 0.08
DEVIATION_RHO = 0.80
MEASUREMENT_NOISE_SD = 0.25

# ------------------------------------------------------------
# Part A: node mixing
# power > 1 => more selective; power < 1 => more mixed
# ------------------------------------------------------------
MIXING_LEVELS = {
    "selective": 3.0,
    "canonical": 1.0,
    "strongly_mixed": 0.45,
}

# ------------------------------------------------------------
# Part B: participant heterogeneity
#
# A single scalar multiplies the same perturbation mechanism.
# 0.0 = identical subject memberships
# 1.0 = original "moderate" setting
# 2.0 = stronger between-subject differences
# ------------------------------------------------------------
HETEROGENEITY_LEVELS = {
    "none": 0.0,
    "moderate": 1.0,
    "strong": 2.0,
}

BASE_PROFILE_SD = 0.22
BASE_LOCAL_CONCENTRATION = 120.0

MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 120
    N_REPS = 3
elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 180
    N_REPS = 20
else:
    raise ValueError("RUN_MODE must be 'quick' or 'paper'")

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("RUN_MODE:", RUN_MODE)
print("subjects:", N_SUBJECTS)
print("timepoints:", T_PER_SUBJECT)
print("replicates / condition:", N_REPS)
print("output:", OUTPUT_DIR)

## 1. MS-AA fitting helpers

These mirror the fitting and normalization logic used in the first
soft-membership notebook and the canonical simulation.

In [ ]:
def coerce_TxV(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D T x V matrix, got {x.shape}")

    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(
        np.float32,
        copy=False,
    )


def build_subject_objects(subject_list):
    return [
        Subject(
            X=coerce_TxV(x),
            sX=coerce_TxV(x),
        )
        for x in subject_list
    ]


def fit_msaa(
    subject_list,
    K,
    seed,
    verbose=False,
):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(
        subject_list
    )

    t0 = time.time()

    (
        results_subj,
        C,
        cost_fun,
        varexpl,
        secs,
    ) = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )

    elapsed = time.time() - t0

    if verbose:
        print(
            f"[MS-AA] K={K} | "
            f"VarExpl={varexpl*100:.2f}% | "
            f"{elapsed:.2f}s"
        )

    return dict(
        results_subj=results_subj,
        C=C,
        cost_fun=cost_fun,
        varexpl=varexpl,
        secs=secs,
        elapsed=elapsed,
    )


def reconstruct_from_archetypes(sub):
    return (
        np.asarray(
            sub["sXC"],
            dtype=float,
        )
        @ np.asarray(
            sub["S"],
            dtype=float,
        )
    )


def normalize_across_subjects(
    subject_list,
    eps=1e-8,
):
    raw = [
        np.asarray(
            x,
            dtype=float,
        )
        for x in subject_list
    ]

    means = [
        x.mean(
            axis=0,
            keepdims=True,
        )
        for x in raw
    ]

    centered = [
        x - m
        for x, m in zip(
            raw,
            means,
        )
    ]

    stacked = np.vstack(
        centered
    )

    across_mu = stacked.mean(
        axis=0,
        keepdims=True,
    )

    sd = (
        stacked.std(
            axis=0,
            keepdims=True,
        )
        + eps
    )

    normalized = [
        (
            (x - across_mu)
            / sd
        ).astype(
            np.float32,
            copy=False,
        )
        for x in centered
    ]

    return normalized, sd

## 2. Canonical overlapping spatial architecture

The six-motif architecture is the same one used in the first soft-membership
extension. We convert the \(K	imes V\) motif matrix into a node-by-motif
composition and manipulate only its entropy.

In [ ]:
def make_overlapping_motifs():
    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    V = (
        n_subnetworks
        * NODES_PER_SUBNETWORK
    )

    motifs = np.full(
        (K_TRUE, V),
        0.05,
        dtype=float,
    )

    motif_subnetworks = [
        [0, 3, 8],
        [1, 3, 9],
        [2, 5, 10],
        [2, 4, 11],
        [0, 6, 7],
        [1, 6, 8],
    ]

    for k, active in enumerate(
        motif_subnetworks
    ):
        for subnet in active:
            lo = (
                subnet
                * NODES_PER_SUBNETWORK
            )

            hi = (
                lo
                + NODES_PER_SUBNETWORK
            )

            motifs[
                k,
                lo:hi,
            ] = 1.0

    return motifs


TRUE_MOTIFS_RAW = make_overlapping_motifs()
V = TRUE_MOTIFS_RAW.shape[1]


def normalize_composition(W):
    W = np.asarray(
        W,
        dtype=float,
    )

    W = np.clip(
        W,
        1e-12,
        None,
    )

    return (
        W
        / W.sum(
            axis=-1,
            keepdims=True,
        )
    )


def change_mixing_strength(
    W,
    power,
):
    return normalize_composition(
        np.asarray(
            W,
            dtype=float,
        ) ** float(power)
    )


def normalized_entropy(W):
    W = normalize_composition(
        W
    )

    H = -np.sum(
        W
        * np.log(
            W + 1e-12
        ),
        axis=-1,
    )

    return (
        H
        / np.log(
            W.shape[-1]
        )
    )


def effective_number(W):
    W = normalize_composition(
        W
    )

    H = -np.sum(
        W
        * np.log(
            W + 1e-12
        ),
        axis=-1,
    )

    return np.exp(H)


W_CANONICAL = normalize_composition(
    TRUE_MOTIFS_RAW.T
)

SIMULATION_TRUTH_ENTROPY = {}

for name, power in MIXING_LEVELS.items():
    W = change_mixing_strength(
        W_CANONICAL,
        power,
    )

    SIMULATION_TRUTH_ENTROPY[name] = float(
        normalized_entropy(W).mean()
    )

    print(
        f"{name:15s} | "
        f"entropy={normalized_entropy(W).mean():.3f} | "
        f"effective motifs/node={effective_number(W).mean():.2f} | "
        f"mean max weight={W.max(axis=1).mean():.3f}"
    )

## 3. Participant heterogeneity with an explicit scale parameter

The first notebook used one arbitrary heterogeneity setting. Here one scalar,
\(h\), controls the same perturbation mechanism.

- \(h=0\): all participants have the same node-composition matrix.
- \(h=1\): reproduces the original moderate setting.
- \(h=2\): increases both global motif preference differences and local
  node-level deviations.

The local Dirichlet concentration is scaled as
\(120/h^2\), so increasing \(h\) produces larger node-level variation while
\(h=0\) remains exactly identical across participants.

In [ ]:
def sample_subject_compositions(
    W_group,
    n_subjects,
    seed,
    heterogeneity_scale,
):
    rng = np.random.default_rng(
        seed
    )

    h = float(
        heterogeneity_scale
    )

    if h == 0:
        return np.repeat(
            W_group[
                None,
                :,
                :
            ],
            n_subjects,
            axis=0,
        )

    profile_sd = (
        BASE_PROFILE_SD
        * h
    )

    local_concentration = (
        BASE_LOCAL_CONCENTRATION
        / (
            h ** 2
        )
    )

    W_subjects = []

    for _ in range(
        n_subjects
    ):
        bias = np.exp(
            rng.normal(
                0,
                profile_sd,
                size=K_TRUE,
            )
        )

        expected = normalize_composition(
            W_group
            * bias[
                None,
                :
            ]
        )

        W_s = np.zeros_like(
            expected
        )

        for v in range(V):
            alpha = np.maximum(
                expected[v]
                * local_concentration,
                1e-3,
            )

            W_s[v] = rng.dirichlet(
                alpha
            )

        W_subjects.append(
            W_s
        )

    return np.stack(
        W_subjects,
        axis=0,
    )

## 4. Metastable temporal process

The temporal generator is unchanged from the first extension. After smooth
state transitions and secondary recruitment, the latent activations receive
time-varying gain and signed autocorrelated deviations, so the observations are
not a noiseless exact MS-AA factorization.

In [ ]:
def sample_state_sequence(
    T,
    K,
    rng,
    mean_dwell=MEAN_DWELL,
):
    states = []
    current = int(
        rng.integers(K)
    )

    while len(states) < T:
        dwell = max(
            2,
            int(
                rng.geometric(
                    1.0
                    / mean_dwell
                )
            ),
        )

        states.extend(
            [current] * dwell
        )

        candidates = [
            k
            for k in range(K)
            if k != current
        ]

        current = int(
            rng.choice(
                candidates
            )
        )

    return np.asarray(
        states[:T],
        dtype=int,
    )


def smooth_state_activations(
    states,
    K,
    transition_width=TRANSITION_WIDTH,
):
    T = len(states)

    A = np.zeros(
        (T, K),
        dtype=float,
    )

    A[
        np.arange(T),
        states,
    ] = 1.0

    if transition_width > 0:
        x = np.arange(
            -transition_width,
            transition_width + 1,
        )

        sigma = max(
            1.0,
            transition_width / 2,
        )

        kernel = np.exp(
            -(
                x ** 2
            )
            / (
                2
                * sigma ** 2
            )
        )

        kernel /= kernel.sum()

        for k in range(K):
            A[:, k] = np.convolve(
                A[:, k],
                kernel,
                mode="same",
            )

    A /= (
        A.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return A


def ar1_process(
    T,
    K,
    rho,
    rng,
):
    x = np.zeros(
        (T, K),
        dtype=float,
    )

    eps = rng.normal(
        0,
        1,
        size=(T, K),
    )

    x[0] = eps[0]

    innovation = np.sqrt(
        max(
            1e-12,
            1.0
            - rho ** 2,
        )
    )

    for t in range(
        1,
        T,
    ):
        x[t] = (
            rho
            * x[t - 1]
            + innovation
            * eps[t]
        )

    return x


def add_secondary_recruitment(
    A,
    rng,
):
    T, K = A.shape

    background = np.exp(
        ar1_process(
            T,
            K,
            AR1_RHO,
            rng,
        )
    )

    background /= (
        background.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    mixed = (
        (
            1.0
            - SECONDARY_WEIGHT
        )
        * A
        + SECONDARY_WEIGHT
        * background
    )

    mixed /= (
        mixed.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return mixed


def add_non_simplex_modulation(
    A,
    rng,
):
    T, K = A.shape

    gain_latent = ar1_process(
        T,
        1,
        DEVIATION_RHO,
        rng,
    )[:, 0]

    gain = np.exp(
        GLOBAL_GAIN_SD
        * gain_latent
        - 0.5
        * GLOBAL_GAIN_SD ** 2
    )

    signed_dev = ar1_process(
        T,
        K,
        DEVIATION_RHO,
        rng,
    )

    return (
        gain[
            :,
            None
        ]
        * A
        + SIGNED_DEVIATION_SD
        * signed_dev
    )


def simulate_soft_membership(
    mixing_level,
    heterogeneity_level,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    W_group = change_mixing_strength(
        W_CANONICAL,
        MIXING_LEVELS[
            mixing_level
        ],
    )

    W_subjects = sample_subject_compositions(
        W_group,
        N_SUBJECTS,
        seed + 100000,
        heterogeneity_scale=(
            HETEROGENEITY_LEVELS[
                heterogeneity_level
            ]
        ),
    )

    subjects = []

    for s in range(
        N_SUBJECTS
    ):
        states = sample_state_sequence(
            T_PER_SUBJECT,
            K_TRUE,
            rng,
        )

        A = smooth_state_activations(
            states,
            K_TRUE,
        )

        A = add_secondary_recruitment(
            A,
            rng,
        )

        A = add_non_simplex_modulation(
            A,
            rng,
        )

        X = (
            A
            @ W_subjects[s].T
        )

        X += rng.normal(
            0,
            MEASUREMENT_NOISE_SD,
            size=X.shape,
        )

        subjects.append(
            X
        )

    return dict(
        subjects=subjects,
        W_group=W_group,
        W_subjects=W_subjects,
        mixing_level=mixing_level,
        heterogeneity_level=heterogeneity_level,
        heterogeneity_scale=(
            HETEROGENEITY_LEVELS[
                heterogeneity_level
            ]
        ),
    )

## 5. Matching and recovery metrics

The archetype permutation is determined once from the group-average spatial
maps and then applied unchanged to every participant.

Two participant-specific metrics are emphasized:

- **profile-deviation correlation**: correlation between true and recovered
  participant deviations after removing the group-average motif profile;
- **pairwise participant-distance correlation**: preservation of the geometry
  of individual differences.

We also measure the *magnitude* of participant heterogeneity. This is essential
for the no-heterogeneity control, where a correlation is undefined but any
recovered between-participant dispersion is directly interpretable as spurious.

In [ ]:
def safe_corr(
    a,
    b,
):
    a = np.asarray(
        a,
        dtype=float,
    ).ravel()

    b = np.asarray(
        b,
        dtype=float,
    ).ravel()

    mask = (
        np.isfinite(a)
        & np.isfinite(b)
    )

    a = a[mask]
    b = b[mask]

    if (
        len(a) < 3
        or np.std(a) < 1e-12
        or np.std(b) < 1e-12
    ):
        return np.nan

    return float(
        np.corrcoef(
            a,
            b,
        )[0, 1]
    )


def pattern_corr_matrix(
    A,
    B,
):
    A = np.asarray(
        A,
        dtype=float,
    )

    B = np.asarray(
        B,
        dtype=float,
    )

    A0 = (
        A
        - A.mean(
            axis=1,
            keepdims=True,
        )
    )

    B0 = (
        B
        - B.mean(
            axis=1,
            keepdims=True,
        )
    )

    A0 /= (
        np.linalg.norm(
            A0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    B0 /= (
        np.linalg.norm(
            B0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return (
        A0
        @ B0.T
    )


def optimal_match(
    true_patterns,
    recovered_patterns,
):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
    )

    (
        true_idx,
        recovered_idx,
    ) = linear_sum_assignment(
        -C
    )

    order = np.full(
        true_patterns.shape[0],
        -1,
        dtype=int,
    )

    for ti, ri in zip(
        true_idx,
        recovered_idx,
    ):
        order[ti] = ri

    return (
        order,
        C[
            true_idx,
            recovered_idx,
        ],
    )


def hellinger_similarity(
    P,
    Q,
):
    P = normalize_composition(
        P
    )

    Q = normalize_composition(
        Q
    )

    d = np.sqrt(
        0.5
        * np.sum(
            (
                np.sqrt(P)
                - np.sqrt(Q)
            ) ** 2,
            axis=-1,
        )
    )

    return (
        1.0
        - d
    )


def participant_profile(W):
    return np.asarray(
        W,
        dtype=float,
    ).mean(
        axis=1,
    )


def profile_dispersion(profile):
    profile = np.asarray(
        profile,
        dtype=float,
    )

    centered = (
        profile
        - profile.mean(
            axis=0,
            keepdims=True,
        )
    )

    return float(
        np.sqrt(
            np.mean(
                centered ** 2
            )
        )
    )


def summarize_recovery(
    W_true,
    W_hat,
):
    W_true = normalize_composition(
        W_true
    )

    W_hat = normalize_composition(
        W_hat
    )

    group_true = W_true.mean(
        axis=0
    )

    group_hat = W_hat.mean(
        axis=0
    )

    profile_true = participant_profile(
        W_true
    )

    profile_hat = participant_profile(
        W_hat
    )

    dev_true = (
        profile_true
        - profile_true.mean(
            axis=0,
            keepdims=True,
        )
    )

    dev_hat = (
        profile_hat
        - profile_hat.mean(
            axis=0,
            keepdims=True,
        )
    )

    node_dev_true = (
        W_true
        - group_true[
            None,
            :,
            :
        ]
    )

    node_dev_hat = (
        W_hat
        - group_hat[
            None,
            :,
            :
        ]
    )

    subject_hellinger = [
        np.mean(
            hellinger_similarity(
                W_true[s],
                W_hat[s],
            )
        )
        for s in range(
            W_true.shape[0]
        )
    ]

    entropy_true = normalized_entropy(
        group_true
    )

    entropy_hat = normalized_entropy(
        group_hat
    )

    d_true = pdist(
        profile_true,
        metric="euclidean",
    )

    d_hat = pdist(
        profile_hat,
        metric="euclidean",
    )

    true_dispersion = profile_dispersion(
        profile_true
    )

    recovered_dispersion = profile_dispersion(
        profile_hat
    )

    return dict(
        group_hellinger=float(
            np.mean(
                hellinger_similarity(
                    group_true,
                    group_hat,
                )
            )
        ),
        group_corr=safe_corr(
            group_true,
            group_hat,
        ),
        dominant_accuracy=float(
            np.mean(
                np.argmax(
                    group_true,
                    axis=1,
                )
                == np.argmax(
                    group_hat,
                    axis=1,
                )
            )
        ),
        subject_hellinger=float(
            np.mean(
                subject_hellinger
            )
        ),
        subject_profile_corr=safe_corr(
            profile_true,
            profile_hat,
        ),
        subject_profile_deviation_corr=safe_corr(
            dev_true,
            dev_hat,
        ),
        subject_node_deviation_corr=safe_corr(
            node_dev_true,
            node_dev_hat,
        ),
        subject_distance_corr=safe_corr(
            d_true,
            d_hat,
        ),
        true_subject_dispersion=true_dispersion,
        recovered_subject_dispersion=(
            recovered_dispersion
        ),
        subject_dispersion_error=float(
            recovered_dispersion
            - true_dispersion
        ),
        true_mean_entropy=float(
            entropy_true.mean()
        ),
        recovered_mean_entropy=float(
            entropy_hat.mean()
        ),
        entropy_corr=safe_corr(
            entropy_true,
            entropy_hat,
        ),
        entropy_bias=float(
            entropy_hat.mean()
            - entropy_true.mean()
        ),
        true_mean_effective_n=float(
            effective_number(
                group_true
            ).mean()
        ),
        recovered_mean_effective_n=float(
            effective_number(
                group_hat
            ).mean()
        ),
    )

## 6. Fit and evaluate one simulation

In [ ]:
def evaluate_one(
    mixing_level,
    heterogeneity_level,
    rep,
    seed,
    verbose=False,
):
    sim = simulate_soft_membership(
        mixing_level,
        heterogeneity_level,
        seed,
    )

    (
        subjects,
        sd,
    ) = normalize_across_subjects(
        sim["subjects"]
    )

    fit = fit_msaa(
        subjects,
        K=K_TRUE,
        seed=seed,
        verbose=verbose,
    )

    # subjects x K x V
    S_subject = np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in fit[
                "results_subj"
            ]
        ],
        axis=0,
    )

    S_group = S_subject.mean(
        axis=0
    )

    # Use the normalized node-space truth only to identify motifs.
    true_group_spatial = (
        sim["W_group"].T
        / sd
    )

    (
        order,
        matched,
    ) = optimal_match(
        true_group_spatial,
        S_group,
    )

    if np.any(
        order < 0
    ):
        raise RuntimeError(
            "Incomplete archetype matching."
        )

    # Apply the same group permutation to every subject.
    W_hat = np.stack(
        [
            S_subject[
                s,
                order,
                :
            ].T
            for s in range(
                N_SUBJECTS
            )
        ],
        axis=0,
    )

    W_hat = normalize_composition(
        W_hat
    )

    metrics = summarize_recovery(
        sim["W_subjects"],
        W_hat,
    )

    X_stack = np.vstack(
        subjects
    )

    X_hat = np.vstack(
        [
            reconstruct_from_archetypes(
                sub
            )
            for sub in fit[
                "results_subj"
            ]
        ]
    )

    mse = float(
        np.mean(
            (
                X_stack
                - X_hat
            ) ** 2
        )
    )

    row = dict(
        mixing_level=mixing_level,
        mixing_power=(
            MIXING_LEVELS[
                mixing_level
            ]
        ),
        heterogeneity_level=(
            heterogeneity_level
        ),
        heterogeneity_scale=(
            HETEROGENEITY_LEVELS[
                heterogeneity_level
            ]
        ),
        rep=int(rep),
        seed=int(seed),
        mean_spatial_match=float(
            np.mean(
                matched
            )
        ),
        mse=mse,
        varexpl=float(
            fit["varexpl"]
        ),
        elapsed_sec=float(
            fit["elapsed"]
        ),
        **metrics,
    )

    details = dict(
        W_true=sim[
            "W_subjects"
        ],
        W_hat=W_hat,
        W_true_group=sim[
            "W_subjects"
        ].mean(
            axis=0
        ),
        W_hat_group=W_hat.mean(
            axis=0
        ),
        profile_true=participant_profile(
            sim[
                "W_subjects"
            ]
        ),
        profile_hat=participant_profile(
            W_hat
        ),
    )

    return (
        row,
        details,
    )

## 7. Restart-safe Part A — node-mixing sweep

Participant heterogeneity is fixed at **moderate**, matching the original quick
analysis.

In [ ]:
def atomic_to_csv(
    df,
    path,
):
    path = Path(
        path
    )

    tmp = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    df.to_csv(
        tmp,
        index=False,
    )

    os.replace(
        tmp,
        path,
    )


def load_checkpoint_table(
    checkpoint_dir,
    pattern,
):
    files = sorted(
        checkpoint_dir.glob(
            pattern
        )
    )

    if not files:
        return pd.DataFrame()

    return pd.concat(
        [
            pd.read_csv(
                p
            )
            for p in files
        ],
        ignore_index=True,
    )


for level in MIXING_LEVELS:
    for rep in range(
        N_REPS
    ):
        job = (
            f"{RUN_MODE}_mix_"
            f"{level}_"
            f"rep{rep:03d}"
        )

        job_csv = (
            MIX_CHECKPOINT_DIR
            / f"{job}.csv"
        )

        detail_npz = (
            DETAIL_DIR
            / f"{job}.npz"
        )

        if job_csv.exists():
            print(
                "SKIP:",
                job,
            )
            continue

        # Same stochastic realization across mixing levels.
        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    9001,
                    rep,
                ]
            ).generate_state(
                1
            )[0]
        )

        print(
            f"RUN mixing={level} | "
            f"rep={rep+1}/{N_REPS} | "
            f"seed={seed}"
        )

        (
            row,
            details,
        ) = evaluate_one(
            mixing_level=level,
            heterogeneity_level=(
                "moderate"
            ),
            rep=rep,
            seed=seed,
            verbose=False,
        )

        atomic_to_csv(
            pd.DataFrame(
                [row]
            ),
            job_csv,
        )

        np.savez_compressed(
            detail_npz,
            **details,
        )

        print(
            "  saved:",
            job_csv.name,
        )


mix_results = (
    load_checkpoint_table(
        MIX_CHECKPOINT_DIR,
        f"{RUN_MODE}_mix_*.csv",
    )
)

if len(
    mix_results
) == 0:
    raise RuntimeError(
        "No mixing simulations found."
    )

mix_results = (
    mix_results
    .sort_values(
        [
            "mixing_level",
            "rep",
        ]
    )
    .reset_index(
        drop=True
    )
)

atomic_to_csv(
    mix_results,
    MIX_RESULTS_CSV,
)

print(
    "\nMixing rows:",
    len(
        mix_results
    ),
)

display(
    mix_results
)

## 8. Restart-safe Part B — participant-heterogeneity sweep

Node mixing is fixed at the **canonical** overlap setting.

The no-heterogeneity condition is especially important: true participant
dispersion is exactly zero, so any recovered dispersion quantifies spurious
individual structure introduced by the fit/noise.

In [ ]:
for het in HETEROGENEITY_LEVELS:
    for rep in range(
        N_REPS
    ):
        job = (
            f"{RUN_MODE}_het_"
            f"{het}_"
            f"rep{rep:03d}"
        )

        job_csv = (
            HET_CHECKPOINT_DIR
            / f"{job}.csv"
        )

        detail_npz = (
            DETAIL_DIR
            / f"{job}.npz"
        )

        if job_csv.exists():
            print(
                "SKIP:",
                job,
            )
            continue

        # Same temporal/noise realization across heterogeneity levels.
        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    12001,
                    rep,
                ]
            ).generate_state(
                1
            )[0]
        )

        print(
            f"RUN heterogeneity={het} | "
            f"rep={rep+1}/{N_REPS} | "
            f"seed={seed}"
        )

        (
            row,
            details,
        ) = evaluate_one(
            mixing_level=(
                "canonical"
            ),
            heterogeneity_level=het,
            rep=rep,
            seed=seed,
            verbose=False,
        )

        atomic_to_csv(
            pd.DataFrame(
                [row]
            ),
            job_csv,
        )

        np.savez_compressed(
            detail_npz,
            **details,
        )

        print(
            "  saved:",
            job_csv.name,
        )


het_results = (
    load_checkpoint_table(
        HET_CHECKPOINT_DIR,
        f"{RUN_MODE}_het_*.csv",
    )
)

if len(
    het_results
) == 0:
    raise RuntimeError(
        "No heterogeneity simulations found."
    )

het_results = (
    het_results
    .sort_values(
        [
            "heterogeneity_level",
            "rep",
        ]
    )
    .reset_index(
        drop=True
    )
)

atomic_to_csv(
    het_results,
    HET_RESULTS_CSV,
)

print(
    "\nHeterogeneity rows:",
    len(
        het_results
    ),
)

display(
    het_results
)

## 9. Compact summaries

The primary quantities to inspect before switching to paper mode are:

### Mixing
- group Hellinger similarity
- group correlation
- entropy bias = recovered − true
- dominant-membership accuracy

### Participant heterogeneity
- true vs recovered participant-profile dispersion
- participant-profile deviation correlation
- pairwise participant-distance correlation
- node-level subject-deviation correlation

In [ ]:
mix_summary_cols = [
    "true_mean_entropy",
    "recovered_mean_entropy",
    "entropy_bias",
    "group_hellinger",
    "group_corr",
    "dominant_accuracy",
    "mean_spatial_match",
    "mse",
]

mix_summary = (
    mix_results
    .groupby(
        "mixing_level"
    )[
        mix_summary_cols
    ]
    .agg(
        [
            "mean",
            "sem",
        ]
    )
    .reindex(
        list(
            MIXING_LEVELS
        )
    )
)

print("NODE-MIXING SUMMARY")
display(
    mix_summary
)


het_summary_cols = [
    "true_subject_dispersion",
    "recovered_subject_dispersion",
    "subject_dispersion_error",
    "subject_profile_deviation_corr",
    "subject_node_deviation_corr",
    "subject_distance_corr",
    "subject_hellinger",
    "group_hellinger",
    "mean_spatial_match",
]

het_summary = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        het_summary_cols
    ]
    .agg(
        [
            "mean",
            "sem",
        ]
    )
    .reindex(
        list(
            HETEROGENEITY_LEVELS
        )
    )
)

print("PARTICIPANT-HETEROGENEITY SUMMARY")
display(
    het_summary
)

## 10. Empirical K=46 anchor

The empirical fit is read before plotting so the simulated entropy range can be
shown directly against the observed node-membership distribution.

This comparison is descriptive. The simulation parameters are **not** fit to
the empirical performance of MS-AA.

In [ ]:
SELECTED_ARCHETYPES = [
    4,
    5,
    6,
    12,
    8,
]

candidate_paths = [
    Path(
        "msaa_flexible_outputs_npz/"
        "spatialAA_across_acrossCond_K46.npz"
    ),
    Path(
        "spatialAA_across_acrossCond_K46.npz"
    ),
    Path(
        "../msaa_flexible_outputs_npz/"
        "spatialAA_across_acrossCond_K46.npz"
    ),
]

empirical_path = next(
    (
        p
        for p in candidate_paths
        if p.exists()
    ),
    None,
)

empirical_anchor = None

if empirical_path is None:
    print(
        "K=46 empirical fit not found. "
        "Empirical anchor will be omitted from the figure."
    )

else:
    fit = np.load(
        empirical_path,
        allow_pickle=True,
    )

    results_subj = (
        fit[
            "results_subj"
        ].tolist()
    )

    # subjects x V x K
    W_emp = np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            ).T
            for sub in results_subj
        ],
        axis=0,
    )

    W_emp = normalize_composition(
        W_emp
    )

    entropy_all = normalized_entropy(
        W_emp
    )

    neff_all = effective_number(
        W_emp
    )

    selected = W_emp[
        :,
        :,
        SELECTED_ARCHETYPES,
    ]

    selected_mass = selected.sum(
        axis=-1
    )

    selected_conditional = (
        selected
        / (
            selected_mass[
                ...,
                None
            ]
            + 1e-12
        )
    )

    selected_entropy = (
        -np.sum(
            selected_conditional
            * np.log(
                selected_conditional
                + 1e-12
            ),
            axis=-1,
        )
        / np.log(
            len(
                SELECTED_ARCHETYPES
            )
        )
    )

    empirical_anchor = dict(
        entropy_all=(
            entropy_all
        ),
        neff_all=(
            neff_all
        ),
        selected_mass=(
            selected_mass
        ),
        selected_entropy=(
            selected_entropy
        ),
        median_entropy=float(
            np.median(
                entropy_all
            )
        ),
        median_effective_n=float(
            np.median(
                neff_all
            )
        ),
        median_selected_mass=float(
            np.median(
                selected_mass
            )
        ),
    )

    pd.DataFrame({
        "normalized_entropy_all46": (
            entropy_all.ravel()
        ),
        "effective_n_all46": (
            neff_all.ravel()
        ),
        "selected5_mass": (
            selected_mass.ravel()
        ),
        "selected5_conditional_entropy": (
            selected_entropy.ravel()
        ),
    }).to_csv(
        OUTPUT_DIR
        / "empirical_K46_node_membership_summary.csv",
        index=False,
    )

    print(
        "Empirical fit:",
        empirical_path,
    )

    print(
        "Median normalized entropy (all 46):",
        empirical_anchor[
            "median_entropy"
        ],
    )

    print(
        "Median effective N (all 46):",
        empirical_anchor[
            "median_effective_n"
        ],
    )

    print(
        "Median selected-five mass:",
        empirical_anchor[
            "median_selected_mass"
        ],
    )

## 11. Supplementary Figure Sx — node mixing and empirical anchor

This figure makes the main methodological result explicit:

- overlapping node participation is recoverable at the group level;
- recovered compositions become more selective than the generating
  compositions as true mixing becomes strong;
- the empirical K=46 entropy distribution can be located relative to the
  simulated range.

In [ ]:
def select_representative(
    df,
    level_col,
    level,
    metric,
):
    sub = df[
        df[
            level_col
        ]
        == level
    ].copy()

    target = sub[
        metric
    ].median()

    idx = (
        sub[
            metric
        ]
        - target
    ).abs().idxmin()

    return sub.loc[
        idx
    ]


mix_rep_row = (
    select_representative(
        mix_results,
        level_col=(
            "mixing_level"
        ),
        level=(
            "canonical"
        ),
        metric=(
            "group_hellinger"
        ),
    )
)

mix_rep_job = (
    f"{RUN_MODE}_mix_"
    f"{mix_rep_row['mixing_level']}_"
    f"rep{int(mix_rep_row['rep']):03d}"
)

mix_rep = np.load(
    DETAIL_DIR
    / f"{mix_rep_job}.npz",
    allow_pickle=True,
)

levels = list(
    MIXING_LEVELS
)

pretty = [
    "Selective",
    "Canonical",
    "Strongly mixed",
]

fig = plt.figure(
    figsize=(
        12.8,
        8.5,
    )
)

gs = fig.add_gridspec(
    2,
    3,
    hspace=0.38,
    wspace=0.32,
)

axes = [
    fig.add_subplot(
        gs[0, 0]
    ),
    fig.add_subplot(
        gs[0, 1]
    ),
    fig.add_subplot(
        gs[0, 2]
    ),
    fig.add_subplot(
        gs[1, 0]
    ),
    fig.add_subplot(
        gs[1, 1]
    ),
    fig.add_subplot(
        gs[1, 2]
    ),
]

(
    axA,
    axB,
    axC,
    axD,
    axE,
    axF,
) = axes

# ------------------------------------------------------------
# A/B — representative true/recovered group composition
# ------------------------------------------------------------
vmax = max(
    mix_rep[
        "W_true_group"
    ].max(),
    mix_rep[
        "W_hat_group"
    ].max(),
)

im = axA.imshow(
    mix_rep[
        "W_true_group"
    ].T,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
    vmin=0,
    vmax=vmax,
)

axB.imshow(
    mix_rep[
        "W_hat_group"
    ].T,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
    vmin=0,
    vmax=vmax,
)

for ax, title in [
    (
        axA,
        "True canonical composition",
    ),
    (
        axB,
        "Recovered MS-AA composition",
    ),
]:
    ax.set_title(
        title,
        fontweight="bold",
    )

    ax.set_xlabel(
        "Node"
    )

    ax.set_ylabel(
        "Motif"
    )

fig.colorbar(
    im,
    ax=[
        axA,
        axB,
    ],
    fraction=0.025,
    pad=0.02,
    label=(
        "Node allocation weight"
    ),
)

# ------------------------------------------------------------
# C — group Hellinger similarity across mixing
# ------------------------------------------------------------
x = np.arange(
    len(
        levels
    )
)

g_mean = (
    mix_results
    .groupby(
        "mixing_level"
    )[
        "group_hellinger"
    ]
    .mean()
    .reindex(
        levels
    )
)

g_sem = (
    mix_results
    .groupby(
        "mixing_level"
    )[
        "group_hellinger"
    ]
    .sem()
    .reindex(
        levels
    )
)

axC.errorbar(
    x,
    g_mean,
    yerr=g_sem,
    marker="o",
    linewidth=2,
    capsize=3,
)

axC.set_ylim(
    0,
    1.02,
)

axC.set_xticks(
    x,
    pretty,
)

axC.set_ylabel(
    "Group Hellinger similarity"
)

axC.set_title(
    "Composition recovery",
    fontweight="bold",
)

# ------------------------------------------------------------
# D — true vs recovered entropy per replicate
# ------------------------------------------------------------
for level, marker in zip(
    levels,
    [
        "o",
        "s",
        "^",
    ],
):
    q = mix_results[
        mix_results[
            "mixing_level"
        ]
        == level
    ]

    axD.scatter(
        q[
            "true_mean_entropy"
        ],
        q[
            "recovered_mean_entropy"
        ],
        s=48,
        marker=marker,
        label=level.replace(
            "_",
            " ",
        ).title(),
    )

axD.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    color="0.5",
)

axD.set_xlim(
    0,
    1,
)

axD.set_ylim(
    0,
    1,
)

axD.set_xlabel(
    "True normalized entropy"
)

axD.set_ylabel(
    "Recovered normalized entropy"
)

axD.set_title(
    "MS-AA tracks mixing but is more selective",
    fontweight="bold",
)

axD.legend(
    frameon=False,
)

# ------------------------------------------------------------
# E — empirical anchor
# ------------------------------------------------------------
if empirical_anchor is not None:
    axE.hist(
        empirical_anchor[
            "entropy_all"
        ].ravel(),
        bins=40,
        density=True,
        alpha=0.65,
    )

    for level, style in zip(
        levels,
        [
            ":",
            "-",
            "--",
        ],
    ):
        axE.axvline(
            SIMULATION_TRUTH_ENTROPY[
                level
            ],
            linestyle=style,
            linewidth=2,
            label=(
                f"{level.replace('_', ' ')} "
                f"({SIMULATION_TRUTH_ENTROPY[level]:.2f})"
            ),
        )

    axE.axvline(
        empirical_anchor[
            "median_entropy"
        ],
        linewidth=2.5,
        label=(
            "Empirical median "
            f"({empirical_anchor['median_entropy']:.2f})"
        ),
    )

    axE.set_xlabel(
        "Normalized node-membership entropy"
    )

    axE.set_ylabel(
        "Density"
    )

    axE.legend(
        frameon=False,
        fontsize=8,
    )

else:
    axE.text(
        0.5,
        0.5,
        "Empirical K=46 fit\nnot available",
        transform=axE.transAxes,
        ha="center",
        va="center",
    )

    axE.set_xticks(
        []
    )

    axE.set_yticks(
        []
    )

axE.set_title(
    "Empirical geometry anchors the simulated range",
    fontweight="bold",
)

# ------------------------------------------------------------
# F — dominant motif recovery
# ------------------------------------------------------------
dom_mean = (
    mix_results
    .groupby(
        "mixing_level"
    )[
        "dominant_accuracy"
    ]
    .mean()
    .reindex(
        levels
    )
)

dom_sem = (
    mix_results
    .groupby(
        "mixing_level"
    )[
        "dominant_accuracy"
    ]
    .sem()
    .reindex(
        levels
    )
)

axF.errorbar(
    x,
    dom_mean,
    yerr=dom_sem,
    marker="o",
    linewidth=2,
    capsize=3,
)

axF.set_ylim(
    0,
    1.02,
)

axF.set_xticks(
    x,
    pretty,
)

axF.set_ylabel(
    "Dominant-membership accuracy"
)

axF.set_title(
    "Preferred motif remains identifiable",
    fontweight="bold",
)

for ax, letter in zip(
    axes,
    list(
        "ABCDEF"
    ),
):
    ax.text(
        -0.10,
        1.05,
        letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )

fig.suptitle(
    "Spatial MS-AA recovers overlapping node composition while favoring a more concentrated representation",
    fontsize=15,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.07,
    right=0.96,
    top=0.90,
    bottom=0.08,
)

MIX_FIG_PNG = (
    OUTPUT_DIR
    / "Supplement_node_mixing_recovery.png"
)

MIX_FIG_PDF = (
    OUTPUT_DIR
    / "Supplement_node_mixing_recovery.pdf"
)

fig.savefig(
    MIX_FIG_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    MIX_FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    MIX_FIG_PNG,
)

print(
    "Saved:",
    MIX_FIG_PDF,
)

## 12. Supplementary Figure Sy — participant heterogeneity

This figure asks a different question from the node-mixing analysis.

The central control is the **none** condition: if no true participant
differences exist, recovered participant dispersion should remain small.

For moderate and strong heterogeneity, the key question is whether MS-AA
preserves the *pattern* of participant deviations and pairwise differences.

In [ ]:
het_rep_row = (
    select_representative(
        het_results,
        level_col=(
            "heterogeneity_level"
        ),
        level=(
            "moderate"
        ),
        metric=(
            "subject_hellinger"
        ),
    )
)

het_rep_job = (
    f"{RUN_MODE}_het_"
    f"{het_rep_row['heterogeneity_level']}_"
    f"rep{int(het_rep_row['rep']):03d}"
)

het_rep = np.load(
    DETAIL_DIR
    / f"{het_rep_job}.npz",
    allow_pickle=True,
)

het_levels = list(
    HETEROGENEITY_LEVELS
)

het_pretty = [
    "None",
    "Moderate",
    "Strong",
]

fig = plt.figure(
    figsize=(
        12.8,
        8.3,
    )
)

gs = fig.add_gridspec(
    2,
    3,
    hspace=0.38,
    wspace=0.32,
)

axes = [
    fig.add_subplot(
        gs[0, 0]
    ),
    fig.add_subplot(
        gs[0, 1]
    ),
    fig.add_subplot(
        gs[0, 2]
    ),
    fig.add_subplot(
        gs[1, 0]
    ),
    fig.add_subplot(
        gs[1, 1]
    ),
    fig.add_subplot(
        gs[1, 2]
    ),
]

(
    axA,
    axB,
    axC,
    axD,
    axE,
    axF,
) = axes

true_profile = (
    het_rep[
        "profile_true"
    ]
)

hat_profile = (
    het_rep[
        "profile_hat"
    ]
)

vmax = max(
    true_profile.max(),
    hat_profile.max(),
)

im = axA.imshow(
    true_profile,
    aspect="auto",
    cmap="viridis",
    vmin=0,
    vmax=vmax,
)

axB.imshow(
    hat_profile,
    aspect="auto",
    cmap="viridis",
    vmin=0,
    vmax=vmax,
)

for ax, title in [
    (
        axA,
        "True participant profiles",
    ),
    (
        axB,
        "Recovered MS-AA profiles",
    ),
]:
    ax.set_title(
        title,
        fontweight="bold",
    )

    ax.set_xlabel(
        "Motif"
    )

    ax.set_ylabel(
        "Participant"
    )

fig.colorbar(
    im,
    ax=[
        axA,
        axB,
    ],
    fraction=0.025,
    pad=0.02,
    label=(
        "Mean node-allocation weight"
    ),
)

# ------------------------------------------------------------
# C — deviations, not raw profiles
# ------------------------------------------------------------
true_dev = (
    true_profile
    - true_profile.mean(
        axis=0,
        keepdims=True,
    )
)

hat_dev = (
    hat_profile
    - hat_profile.mean(
        axis=0,
        keepdims=True,
    )
)

dev_r = safe_corr(
    true_dev,
    hat_dev,
)

axC.scatter(
    true_dev.ravel(),
    hat_dev.ravel(),
    s=30,
    alpha=0.65,
)

lo = min(
    true_dev.min(),
    hat_dev.min(),
)

hi = max(
    true_dev.max(),
    hat_dev.max(),
)

axC.plot(
    [
        lo,
        hi,
    ],
    [
        lo,
        hi,
    ],
    linestyle="--",
    color="0.5",
)

axC.axhline(
    0,
    color="0.8",
    linewidth=1,
)

axC.axvline(
    0,
    color="0.8",
    linewidth=1,
)

axC.set_xlabel(
    "True deviation from group profile"
)

axC.set_ylabel(
    "Recovered deviation"
)

axC.set_title(
    "Individual deviations\n"
    f"r = {dev_r:.2f}",
    fontweight="bold",
)

# ------------------------------------------------------------
# D — magnitude of heterogeneity
# ------------------------------------------------------------
x = np.arange(
    len(
        het_levels
    )
)

true_disp = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "true_subject_dispersion"
    ]
    .mean()
    .reindex(
        het_levels
    )
)

true_disp_sem = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "true_subject_dispersion"
    ]
    .sem()
    .reindex(
        het_levels
    )
)

hat_disp = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "recovered_subject_dispersion"
    ]
    .mean()
    .reindex(
        het_levels
    )
)

hat_disp_sem = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "recovered_subject_dispersion"
    ]
    .sem()
    .reindex(
        het_levels
    )
)

axD.errorbar(
    x,
    true_disp,
    yerr=true_disp_sem,
    marker="o",
    linewidth=2,
    capsize=3,
    label="True",
)

axD.errorbar(
    x,
    hat_disp,
    yerr=hat_disp_sem,
    marker="s",
    linewidth=2,
    capsize=3,
    label="MS-AA",
)

axD.set_xticks(
    x,
    het_pretty,
)

axD.set_ylabel(
    "Participant-profile dispersion"
)

axD.set_title(
    "Does MS-AA recover heterogeneity magnitude?",
    fontweight="bold",
)

axD.legend(
    frameon=False,
)

# ------------------------------------------------------------
# E — profile-deviation correlation
# ------------------------------------------------------------
dev_mean = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "subject_profile_deviation_corr"
    ]
    .mean()
    .reindex(
        het_levels
    )
)

dev_sem = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "subject_profile_deviation_corr"
    ]
    .sem()
    .reindex(
        het_levels
    )
)

axE.errorbar(
    x,
    dev_mean,
    yerr=dev_sem,
    marker="o",
    linewidth=2,
    capsize=3,
)

axE.axhline(
    0,
    linestyle=":",
    color="0.6",
)

axE.set_xticks(
    x,
    het_pretty,
)

axE.set_ylim(
    -1,
    1,
)

axE.set_ylabel(
    "Correlation of participant deviations"
)

axE.set_title(
    "Recovery of individual profile pattern",
    fontweight="bold",
)

# ------------------------------------------------------------
# F — pairwise participant geometry
# ------------------------------------------------------------
dist_mean = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "subject_distance_corr"
    ]
    .mean()
    .reindex(
        het_levels
    )
)

dist_sem = (
    het_results
    .groupby(
        "heterogeneity_level"
    )[
        "subject_distance_corr"
    ]
    .sem()
    .reindex(
        het_levels
    )
)

axF.errorbar(
    x,
    dist_mean,
    yerr=dist_sem,
    marker="o",
    linewidth=2,
    capsize=3,
)

axF.axhline(
    0,
    linestyle=":",
    color="0.6",
)

axF.set_xticks(
    x,
    het_pretty,
)

axF.set_ylim(
    -1,
    1,
)

axF.set_ylabel(
    "Correlation of pairwise subject distances"
)

axF.set_title(
    "Recovery of participant geometry",
    fontweight="bold",
)

for ax, letter in zip(
    axes,
    list(
        "ABCDEF"
    ),
):
    ax.text(
        -0.10,
        1.05,
        letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="top",
    )

fig.suptitle(
    "Participant-specific MS-AA weights are evaluated against controlled levels of true heterogeneity",
    fontsize=15,
    fontweight="bold",
    y=0.99,
)

fig.subplots_adjust(
    left=0.07,
    right=0.96,
    top=0.90,
    bottom=0.08,
)

HET_FIG_PNG = (
    OUTPUT_DIR
    / "Supplement_subject_heterogeneity_recovery.png"
)

HET_FIG_PDF = (
    OUTPUT_DIR
    / "Supplement_subject_heterogeneity_recovery.pdf"
)

fig.savefig(
    HET_FIG_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    HET_FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    HET_FIG_PNG,
)

print(
    "Saved:",
    HET_FIG_PDF,
)

## 13. Interpretation guide

### If the node-mixing result resembles the quick run

The strongest defensible conclusion is:

> Spatial MS-AA recovered the organization of overlapping node participation
> across a broad range of true mixing, although fitted compositions became more
> concentrated than the generating memberships as overlap increased.

That is more informative than claiming exact recovery of the mixture weights.

### If the empirical entropy remains near the canonical simulation

A defensible motivation is:

> The baseline simulated overlap regime was defined independently of method
> performance. Its normalized node-membership entropy was comparable to that
> observed in the empirical K=46 spatial-MS-AA solution, providing a
> descriptive geometric anchor for the simulated range.

This is **not** evidence that the simulation generated the empirical data.

### Participant-specific claim

Only make a strong participant-specific claim if the heterogeneity sweep shows:

1. little recovered dispersion when the true dispersion is zero;
2. increasing recovered dispersion as true heterogeneity increases;
3. positive recovery of participant-deviation structure and/or pairwise
   participant geometry at moderate or strong heterogeneity.

If those conditions are not met, keep the empirical subject-specific weights
as a useful model output but do not claim the simulation establishes accurate
individual-difference recovery.

### Relationship to PCA

This notebook does not try to construct a forced subject-specific PCA
"membership" analogue. The canonical simulation remains the clean head-to-head
PCA/MS-AA benchmark for motif recovery, latent-span preservation, and
reconstruction. This extension asks what the **simplex-constrained,
participant-specific spatial coefficients of MS-AA** can represent and recover.